# CrewAI Multi-Agent Email Coordination

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/crewai_multi_agent_crew.ipynb)

Build a crew of 3 agents that coordinate through email:
- **Triage agent** — reads inbound emails, classifies type, routes to specialists
- **Specialist agent** — handles the actual request (billing, technical, general)
- **QA agent** — reviews the reply before sending for tone and accuracy

Each agent gets its own inbox. Communication between agents happens via email threads — creating a full audit trail.

**Architecture:**
```
Customer email → triage@company.com (Triage Agent)
                      ↓ routes to
              billing@company.com (Billing Agent)  or
              tech@company.com (Tech Support Agent)
                      ↓ reply reviewed by
              qa@company.com (QA Agent)
                      ↓ approved reply sent
              Customer receives reply in original thread
```

**Prerequisites:**
- [Commune API key](https://commune.email) (free tier)
- [OpenAI API key](https://platform.openai.com)

In [ ]:
!pip install commune-mail crewai crewai-tools langchain-openai -q
print("\u2713 Dependencies installed")

## Setup: Create an Inbox for Each Agent

Each agent in the crew gets its own Commune inbox. This gives every agent:
- A dedicated email address that external parties can reach
- An isolated thread history for audit and debugging
- A separate channel for agent-to-agent communication

In a production deployment you would create these inboxes once and store the IDs in environment variables.

In [ ]:
import os
from commune import CommuneClient

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-your_key_here")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # crewai reads from env

commune = CommuneClient(api_key=COMMUNE_API_KEY)

# Provision one inbox per agent role
triage_inbox = commune.inboxes.create(local_part="crew-triage")
billing_inbox = commune.inboxes.create(local_part="crew-billing")
tech_inbox = commune.inboxes.create(local_part="crew-tech")
qa_inbox = commune.inboxes.create(local_part="crew-qa")

print("\u2713 Agent inboxes provisioned:")
print(f"  Triage:  {triage_inbox.address}  (ID: {triage_inbox.id})")
print(f"  Billing: {billing_inbox.address}  (ID: {billing_inbox.id})")
print(f"  Tech:    {tech_inbox.address}  (ID: {tech_inbox.id})")
print(f"  QA:      {qa_inbox.address}  (ID: {qa_inbox.id})")

## Define CrewAI Tools

We wrap Commune operations as `BaseTool` subclasses for CrewAI. Each agent will be given a subset of these tools — the triage agent only needs to read and classify; the specialist agents need to draft replies; the QA agent needs to review and approve.

In [ ]:
from typing import Type
from pydantic import BaseModel, Field
from crewai.tools import BaseTool


class ReadInboxInput(BaseModel):
    inbox_id: str = Field(description="The Commune inbox ID to read threads from.")
    limit: int = Field(default=10, description="Maximum number of threads to return.")


class ReadInboxTool(BaseTool):
    name: str = "read_inbox"
    description: str = (
        "List the most recent email threads in a Commune inbox. "
        "Returns thread subjects, senders, and message counts."
    )
    args_schema: Type[BaseModel] = ReadInboxInput

    def _run(self, inbox_id: str, limit: int = 10) -> str:
        threads = commune.threads.list(inbox_id=inbox_id, limit=limit)
        if not threads.data:
            return "No threads in inbox."
        lines = []
        for t in threads.data:
            lines.append(
                f"Thread ID: {t.thread_id} | Subject: {t.subject or '(no subject)'} "
                f"| Messages: {t.message_count}"
            )
        return "\n".join(lines)


class SendEmailInput(BaseModel):
    to: str = Field(description="Recipient email address.")
    subject: str = Field(description="Email subject line.")
    body: str = Field(description="Plain-text email body.")
    inbox_id: str = Field(description="The Commune inbox ID to send from.")
    thread_id: str = Field(default="", description="Thread ID to reply within (empty for new thread).")


class SendEmailTool(BaseTool):
    name: str = "send_email"
    description: str = (
        "Send an email from a Commune inbox. If thread_id is provided, "
        "the message appears as a reply in that thread."
    )
    args_schema: Type[BaseModel] = SendEmailInput

    def _run(self, to: str, subject: str, body: str, inbox_id: str, thread_id: str = "") -> str:
        kwargs = {"to": to, "subject": subject, "text": body, "inbox_id": inbox_id}
        if thread_id:
            kwargs["thread_id"] = thread_id
        result = commune.messages.send(**kwargs)
        return f"Email sent. Message ID: {result.get('message_id', 'ok')}"


class SearchThreadsInput(BaseModel):
    query: str = Field(description="Natural language search query.")
    inbox_id: str = Field(description="The Commune inbox ID to search within.")
    limit: int = Field(default=5, description="Maximum number of results.")


class SearchThreadsTool(BaseTool):
    name: str = "search_threads"
    description: str = (
        "Search email threads using semantic (natural language) search. "
        "Results are ranked by similarity, not keyword overlap."
    )
    args_schema: Type[BaseModel] = SearchThreadsInput

    def _run(self, query: str, inbox_id: str, limit: int = 5) -> str:
        results = commune.search.threads(query=query, inbox_id=inbox_id, limit=limit)
        if not results:
            return "No matching threads found."
        lines = []
        for r in results:
            lines.append(f"Thread ID: {r.thread_id} | Subject: {r.subject} | Score: {r.score:.2f}")
        return "\n".join(lines)


# Instantiate tools
read_inbox_tool = ReadInboxTool()
send_email_tool = SendEmailTool()
search_threads_tool = SearchThreadsTool()

print("\u2713 CrewAI tools ready")

## Define the Crew Agents

Each CrewAI `Agent` gets a role, a goal, a backstory, and a set of tools. The combination of role + backstory shapes the agent's behavior even before it receives a task.

Note that we set `verbose=True` so you can follow the reasoning in this notebook.

In [ ]:
from crewai import Agent

triage_agent = Agent(
    role="Support Triage Specialist",
    goal=(
        "Read inbound customer emails, classify the issue type "
        "(billing, technical, general, feature_request, urgent), "
        "and produce a structured triage report for the specialist agent."
    ),
    backstory=(
        "You are an experienced support triage specialist. You've seen thousands "
        "of customer issues and can immediately identify the category and urgency "
        "of any request. You are concise and precise in your assessments."
    ),
    tools=[read_inbox_tool, search_threads_tool],
    verbose=True,
    allow_delegation=False,
)

specialist_agent = Agent(
    role="Senior Support Specialist",
    goal=(
        "Given a triage report, draft a complete, empathetic, and accurate "
        "reply to the customer's issue. Search thread history for context "
        "before drafting the reply."
    ),
    backstory=(
        "You are a senior customer support specialist with deep product knowledge. "
        "You excel at writing clear, empathetic responses that resolve issues on the first reply. "
        "You always acknowledge the customer's frustration before providing a solution."
    ),
    tools=[search_threads_tool],
    verbose=True,
    allow_delegation=False,
)

qa_agent = Agent(
    role="Quality Assurance Reviewer",
    goal=(
        "Review a drafted customer support reply for tone, accuracy, completeness, "
        "and professionalism. Either approve the reply (with minor edits) or "
        "reject it with specific feedback."
    ),
    backstory=(
        "You are a QA reviewer who ensures every customer-facing email meets the "
        "highest standards. You check for empathy, factual accuracy, clear next steps, "
        "and brand voice. You are thorough but not overly critical — if a draft is good, "
        "you approve it quickly with only necessary edits."
    ),
    tools=[send_email_tool],
    verbose=True,
    allow_delegation=False,
)

print("\u2713 3 agents defined: Triage, Specialist, QA")

## Define Crew Tasks

Tasks define what each agent must produce. The output of each task is passed to the next as context — creating a sequential pipeline.

Note: `expected_output` is a string description of what a correct output looks like. CrewAI uses this to evaluate whether the agent has completed the task.

In [ ]:
from crewai import Task

def build_crew_tasks(email_payload: dict) -> list:
    """Build the task pipeline for a given inbound email payload."""

    sender = email_payload["sender"]
    subject = email_payload["subject"]
    content = email_payload["content"]
    thread_id = email_payload["thread_id"]
    inbox_id = email_payload["inbox_id"]

    triage_task = Task(
        description=(
            f"A new customer email has arrived.\n\n"
            f"From: {sender}\n"
            f"Subject: {subject}\n"
            f"Thread ID: {thread_id}\n"
            f"Inbox ID: {inbox_id}\n\n"
            f"Email content:\n{content}\n\n"
            f"Search the inbox for any prior threads from {sender} or about '{subject}'. "
            f"Then produce a structured triage report with: issue_type, urgency (low/medium/high/critical), "
            f"prior_context (summary of any past threads, or 'none'), and routing (billing/technical/general)."
        ),
        expected_output=(
            "A structured triage report with fields: issue_type, urgency, prior_context, routing, "
            "and a one-sentence summary of the customer's issue."
        ),
        agent=triage_agent,
    )

    specialist_task = Task(
        description=(
            f"Using the triage report from the previous step, draft a complete email reply "
            f"for the customer at {sender}.\n\n"
            f"The reply must:\n"
            f"- Acknowledge the specific issue (not a generic template)\n"
            f"- Provide a concrete solution or next step\n"
            f"- Reference any relevant prior context found by the triage agent\n"
            f"- Be warm, professional, and under 150 words\n\n"
            f"Do NOT send the email yet — just produce the draft text."
        ),
        expected_output=(
            "A complete email reply draft: subject line and body text, "
            "ready for QA review. Under 150 words."
        ),
        agent=specialist_agent,
        context=[triage_task],
    )

    qa_task = Task(
        description=(
            f"Review the draft reply from the specialist agent.\n\n"
            f"Check for:\n"
            f"- Empathetic tone (does it acknowledge the customer's frustration?)\n"
            f"- Factual accuracy (no false promises or incorrect information)\n"
            f"- Clear next step (does the customer know what to do?)\n"
            f"- Appropriate length (under 150 words)\n\n"
            f"If the draft passes QA, send the approved reply using the send_email tool:\n"
            f"  to: {sender}\n"
            f"  inbox_id: {inbox_id}\n"
            f"  thread_id: {thread_id}\n\n"
            f"If the draft fails QA, return specific feedback (do not send)."
        ),
        expected_output=(
            "Either: 'APPROVED — reply sent' with the final text, "
            "or: 'REJECTED — [specific feedback]' if the draft needs revision."
        ),
        agent=qa_agent,
        context=[triage_task, specialist_task],
    )

    return [triage_task, specialist_task, qa_task]


print("\u2713 Task builder ready")

## Assemble and Run the Crew

The `Crew` object wires the agents and tasks together. `Process.sequential` means each task must complete before the next starts — the output of each task is passed as context.

We simulate a high-urgency billing complaint to test the full pipeline.

In [ ]:
from crewai import Crew, Process

# Simulate an inbound email payload (same structure as a Commune webhook)
simulated_email = {
    "sender": "bob@example.com",
    "subject": "Charged twice this month — need immediate refund",
    "content": (
        "I was charged $99 twice this month (June 3 and June 5). "
        "I've been a customer for 2 years and this has never happened before. "
        "Please refund the duplicate charge immediately. Order numbers: #4521 and #4522."
    ),
    "thread_id": "thread_demo_billing_001",
    "inbox_id": triage_inbox.id,
}

tasks = build_crew_tasks(simulated_email)

crew = Crew(
    agents=[triage_agent, specialist_agent, qa_agent],
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
)

print("Starting crew...")
print("=" * 60)
result = crew.kickoff()
print("=" * 60)
print("\nFinal crew output:")
print(result)

## View the Thread History

After the crew runs, you can inspect the thread to verify the reply was sent in the correct thread and the full conversation is intact.

In [ ]:
# List threads in the triage inbox to confirm activity
threads = commune.threads.list(inbox_id=triage_inbox.id, limit=5)

print(f"Threads in triage inbox ({triage_inbox.address}):")
for t in threads.data:
    print(f"  [{t.thread_id}] {t.subject or '(no subject)'} — {t.message_count} message(s)")

# If the crew sent a reply, it will appear in the thread
thread_id = simulated_email["thread_id"]
print(f"\nLooking for thread: {thread_id}")
matching = [t for t in threads.data if t.thread_id == thread_id]
if matching:
    msgs = commune.threads.messages(thread_id)
    print(f"Found {len(msgs)} message(s) in thread:")
    for m in msgs:
        print(f"  [{m.direction}] {m.content[:120]}...")
else:
    print("Thread not yet visible — reply may not have been sent in simulation mode.")

## What's next?

- **Add a 4th agent** — an escalation agent that sends an SMS alert for critical issues via `commune.sms.send()`
- **Parallel process** — switch from `Process.sequential` to `Process.hierarchical` with a manager LLM for more complex routing
- **Webhook integration** — connect a Flask webhook so the crew runs automatically on every inbound email
- **[LangChain notebook](./langchain_customer_support.ipynb)** — single-agent support with LangChain tools
- **[OpenAI Agents notebook](./openai_agents_email.ipynb)** — OpenAI Agents SDK function tools

**CrewAI docs:** https://docs.crewai.com  
**Commune docs:** https://commune.email/docs